
# Phase 2: ML Foundation — Model Training

**Phase doc:** `docs/phases/phase-02-ml-foundation.md`  
**Depends on:** `ml/eda.ipynb` complete — feature list, encoding, and imputation strategies all decided  
**Outputs:** `ml/artifacts/model.pkl`, `ml/artifacts/training_stats.json`

---

## Training Summary
*(Fill in after all sections are complete)*

| Item | Value |
|------|-------|
| Features used | 12 (4 required + 8 optional) |
| Outliers removed | 2 rows (`GrLivArea > 4000 AND SalePrice < $200k`) |
| Target transform | `np.log1p()` — inverted with `np.expm1()` at inference |
| Baseline MAE (DummyRegressor) | *(fill in after Section 3)* |
| Final model | *(fill in after Section 5)* |
| Test MAE | *(fill in after Section 5)* |
| Test RMSE | *(fill in after Section 5)* |
| Test R² | *(fill in after Section 5)* |

---

**Leakage rule:** All preprocessing statistics are computed on training data only and stored inside the `sklearn.Pipeline`. The test set is not touched until final evaluation.



---
## Section 1: Imports and Data Loading


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
sns.set_theme(style="whitegrid")

DATA_PATH = "data/ames.csv"
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")


Loaded: 1,460 rows × 81 columns



---
## Section 2: Split + Outlier Removal

**Split first** (ML rule) — then remove outliers from the training set only.  
The 2 partial-interest outliers (`GrLivArea > 4000 AND SalePrice < $200k`) are dropped from `X_train`/`y_train` — the test set is not touched.


In [2]:

# --- Feature list locked from Phase 1 EDA ---
FEATURES = [
    # Required
    "GrLivArea", "OverallQual", "YearBuilt", "Neighborhood",
    # Optional
    "TotalBsmtSF", "GarageCars", "FullBath", "YearRemodAdd",
    "Fireplaces", "LotArea", "MasVnrArea", "Exterior1st",
]

NUMERIC_FEATURES = [
    "GrLivArea", "OverallQual", "YearBuilt",
    "TotalBsmtSF", "GarageCars", "FullBath", "YearRemodAdd",
    "Fireplaces", "LotArea", "MasVnrArea",
]

CATEGORICAL_FEATURES = ["Neighborhood", "Exterior1st"]

# --- Split (80/20, random_state=42 — same seed as EDA) ---
X = df[FEATURES]
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Before outlier removal — train: {len(X_train):,}  test: {len(X_test):,}")

# --- Remove 2 outliers from training set only ---
# These are partial-interest sales (SaleCondition = "Partial") identified in EDA Section 7
outlier_mask = (X_train["GrLivArea"] > 4000) & (y_train < 200_000)
n_removed = outlier_mask.sum()
X_train = X_train[~outlier_mask]
y_train = y_train[~outlier_mask]

print(f"Outliers removed from train: {n_removed}")
print(f"After outlier removal  — train: {len(X_train):,}  test: {len(X_test):,}")
print("\nTest set untouched.")


Before outlier removal — train: 1,168  test: 292
Outliers removed from train: 2
After outlier removal  — train: 1,166  test: 292

Test set untouched.



---
## Section 3: Preprocessing Pipeline + Baseline (DummyRegressor)

**Preprocessing decisions from EDA:**

| Feature group | Strategy |
|--------------|----------|
| Numeric missing | `SimpleImputer(strategy="median")` — fit on train only |
| `YearRemodAdd` missing | Handled by median imputer (approximation); exact `YearBuilt` fill is done in production pipeline |
| `MasVnrArea` missing | Imputed with `0` — NA means no veneer (Group A) |
| `Neighborhood` | `TargetEncoder` — fit on train only |
| `Exterior1st` | Rare values (<10 rows) → `"Other"`, then `OneHotEncoder` |

**Note on TargetEncoder:** Available in scikit-learn ≥ 1.3 as `sklearn.preprocessing.TargetEncoder`. Uses leave-one-out encoding internally to avoid leakage within the training fold.


In [3]:

from sklearn.preprocessing import TargetEncoder

# --- Rare-value binning for Exterior1st (values with <10 rows in train → "Other") ---
ext_counts = X_train["Exterior1st"].value_counts()
rare_exteriors = ext_counts[ext_counts < 10].index.tolist()
print(f"Exterior1st rare values (< 10 rows) to bin as 'Other': {rare_exteriors}")

X_train = X_train.copy()
X_test = X_test.copy()
X_train["Exterior1st"] = X_train["Exterior1st"].replace(rare_exteriors, "Other")
X_test["Exterior1st"] = X_test["Exterior1st"].replace(rare_exteriors, "Other")

# --- MasVnrArea: fill NA with 0 (Group A — no veneer means area is 0) ---
X_train["MasVnrArea"] = X_train["MasVnrArea"].fillna(0)
X_test["MasVnrArea"] = X_test["MasVnrArea"].fillna(0)

# --- Log-transform target (decided in EDA Section 5) ---
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# --- Preprocessing pipeline ---
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

neighborhood_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target_enc", TargetEncoder(random_state=42)),
])

preprocessor = ColumnTransformer([
    ("num",   numeric_transformer,      NUMERIC_FEATURES),
    ("cat",   categorical_transformer,  ["Exterior1st"]),
    ("nbhd",  neighborhood_transformer, ["Neighborhood"]),
])

# --- Baseline: DummyRegressor (predicts training median for every input) ---
baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="median")),
])

# Fit on log-scale target (consistent with how the final model will be trained)
baseline_pipeline.fit(X_train, y_train_log)

# Predict and invert log transform
y_pred_baseline_log = baseline_pipeline.predict(X_test)
y_pred_baseline = np.expm1(y_pred_baseline_log)

baseline_mae  = mean_absolute_error(y_test, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
baseline_r2   = r2_score(y_test, y_pred_baseline)

print("=== Baseline: DummyRegressor (median) ===")
print(f"  MAE  : ${baseline_mae:,.0f}")
print(f"  RMSE : ${baseline_rmse:,.0f}")
print(f"  R²   : {baseline_r2:.4f}")
print()
print("This is the floor. The final model must beat this clearly.")


Exterior1st rare values (< 10 rows) to bin as 'Other': ['BrkComm', 'ImStucc', 'CBlock', 'AsphShn', 'Stone']
=== Baseline: DummyRegressor (median) ===
  MAE  : $59,568
  RMSE : $88,667
  R²   : -0.0250

This is the floor. The final model must beat this clearly.



### Baseline Decision (Q2)

| Metric | Baseline Value | Notes |
|--------|---------------|-------|
| MAE | **$59,568** | Average error in USD when predicting the training median for every house |
| RMSE | **$88,667** | |
| R² | **-0.0250** | Slightly negative — expected when using median (not mean) as the constant prediction |

**Q2 — Baseline MAE: $59,568**  
Any real model must beat this by a meaningful margin. Target: MAE < $30,000 (>49% reduction), R² > 0.85.

**Rare Exterior1st values binned to "Other":** `BrkComm`, `ImStucc`, `CBlock`, `AsphShn`, `Stone` (5 values with <10 training rows).
